In [ ]:
import warnings
warnings.filterwarnings("ignore")
import ast
import numpy as np
import pandas as pd
from tqdm import tqdm
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from keras.models import Sequential
from keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import pearsonr

def create_dataset(data, look_back=8):
    X, Y = [], []
    for i in range(len(data)-look_back-1):
        X.append(data[i:(i+look_back), 0])
        Y.append(data[i + look_back, 0])
    return np.array(X), np.array(Y)

# Load ESG data
with open('/content/PaperReady_numeric_esg.txt', 'r') as f:
    content = f.read()
    start = content.find('{')
    end = content.rfind('}') + 1
    numeric_esg = ast.literal_eval(content[start:end])

company_results = []
look_back = 8

for company, values in tqdm(numeric_esg.items(), desc="Hybrid Model"):
    series = pd.Series(values).dropna()
    if len(series) < 20:
        continue

    split_idx = int(len(series) * 0.75)
    train, test = series[:split_idx], series[split_idx:]

    try:
        # ARIMA
        arima = ARIMA(train, order=(0,1,0))
        arima_fit = arima.fit()
        arima_forecast = arima_fit.forecast(steps=len(test))

        # Residuals
        residuals = pd.Series(arima_fit.resid[1:])
        scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_residuals = scaler.fit_transform(residuals.values.reshape(-1,1))
        X, Y = create_dataset(scaled_residuals, look_back)
        X = np.reshape(X, (X.shape[0], X.shape[1], 1))

        # LSTM
        model = Sequential()
        model.add(LSTM(50, input_shape=(look_back, 1)))
        model.add(Dense(1))
        model.compile(loss='mean_squared_error', optimizer='adam')
        model.fit(X, Y, epochs=20, batch_size=1, verbose=0)

        # Forecast residuals
        test_inputs = scaled_residuals[-look_back:].reshape(-1,1)
        lstm_residuals = []
        for _ in range(len(test)):
            x_input = test_inputs[-look_back:].reshape(1, look_back, 1)
            pred = model.predict(x_input, verbose=0)
            test_inputs = np.append(test_inputs, pred)
            lstm_residuals.append(pred[0][0])

        lstm_residuals = scaler.inverse_transform([lstm_residuals])[0]
        hybrid_forecast = arima_forecast + lstm_residuals

        # Metrics
        mse = mean_squared_error(test, hybrid_forecast)
        mae = mean_absolute_error(test, hybrid_forecast)
        mape = np.mean(np.abs((test - hybrid_forecast) / np.where(test == 0, np.nan, test))) * 100
        theils_u = np.sqrt(np.sum((hybrid_forecast - test) ** 2)) / (
            np.sqrt(np.sum(hybrid_forecast ** 2)) + np.sqrt(np.sum(test ** 2))
        )
        bias = np.mean(hybrid_forecast - test)

        company_results.append({
            "Company": company,
            "MSE": round(mse, 4),
            "MAE": round(mae, 4),
            "MAPE (%)": round(mape, 4),
            "Theil's U": round(theils_u, 4),
            "Bias": round(bias, 4),
            "Actual": test.tolist(),
            "ARIMA": arima_forecast.tolist(),
            "LSTM_Residuals": lstm_residuals.tolist(),
            "Hybrid": hybrid_forecast.tolist()
        })

    except Exception as e:
        print(f"Error for {company}: {e}")
        continue

# Final Results
results_df = pd.DataFrame(company_results)

print("\nHybrid Model Performance:")
print(f"✅ Using sliding window of size {look_back} and predicting the next value in each step.\n")

# Only selected metrics
metrics = ["MSE", "MAE", "MAPE (%)", "Theil's U", "Bias"]
avg_metrics = {}

for metric in metrics:
    avg = np.nanmean(results_df[metric])
    avg_metrics[metric] = avg
    print(f"Average {metric}: {round(avg, 4)}")

# Save only selected columns + forecast data
results_df = results_df[["Company"] + metrics + ["Actual", "ARIMA", "LSTM_Residuals", "Hybrid"]]
results_df.to_excel("hybrid_arima_lstm_results_window8.xlsx", index=False)

Hybrid Model: 100%|██████████| 495/495 [1:05:38<00:00,  7.96s/it]



Hybrid Model Performance:
✅ Using sliding window of size 8 and predicting the next value in each step.

Average MSE: 164.512
Average MAE: 7.3564
Average MAPE (%): 39.6394
Average Theil's U: 0.1733
Average Bias: 1.8833
